In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import os

In [2]:
# Create a function that connects to a duckdb file saves the csv files into tables within the database

# To view the tables in actual database file then load the dataset using DuckDB
path = "query/query_data/"
db_path = "analytics.duckdb"  # Path to your persistent database file

def create_duckdb_database(path, db_file="analytics.duckdb"):
    # 1. Connect to (or create) the persistent database file
    conn = duckdb.connect(db_file)

    # Check if folder exists
    if not os.path.exists(path):
        raise FileNotFoundError(f"Directory '{path}' not found.")

    files = [f for f in os.listdir(path) if f.endswith(".csv")]

    for file in files:
        raw_name = os.path.splitext(file)[0]

        # Clean table name
        table_name = "".join(c if c.isalnum() else "_" for c in raw_name)
        file_path = os.path.join(path, file).replace("\\", "/")

        # 2. CREATE TABLE persists the data into the .duckdb file
        conn.execute(f'CREATE OR REPLACE TABLE "{table_name}" AS SELECT * FROM read_csv_auto("{file_path}")')

        print(f"Persisted table in database: {table_name}")

    # 3. Close the connection to flush changes to disk
    conn.close()
    print(f"\nDatabase successfully saved to '{db_file}'")


In [3]:
# Here we run the function
create_duckdb_database(path, db_path)

# --- Test Querying from the DataBase File ---
conn = duckdb.connect(db_path, read_only = True)
print(conn.execute('SELECT COUNT(*) AS retail_sales FROM "auto_sales_2022"').df())
# conn.close()

Persisted table in database: auto_sales_2022
Persisted table in database: kob_industry_total_sales
Persisted table in database: outliers
Persisted table in database: sales_above_10b
Persisted table in database: sale_distribution_naics
Persisted table in database: seasonality_mom
Persisted table in database: top_industry_mom_2019
Persisted table in database: top_industry_mom_2020
Persisted table in database: top_industry_mom_2021
Persisted table in database: top_industry_mom_2022

Database successfully saved to 'analytics.duckdb'
   retail_sales
0             6


### **Create a function that combines the csv files into specific table names**

In [6]:
# Connect to your saved database file
conn = duckdb.connect("analytics.duckdb", read_only=True)

# Modify get_tables to accept the connection object
def get_tables(conn, table_names):
    # Use conn.execute() instead of duckdb.sql()
    return {name: conn.execute(f'SELECT * FROM "{name}"').df() for name in table_names}

# Pass your connection into the function
list_tables = [
    'auto_sales_2022', 'kob_industry_total_sales', 'outliers',
    'sales_above_10b', 'sale_distribution_naics', 'seasonality_mom',
    'top_industry_mom_2019', 'top_industry_mom_2020',
    'top_industry_mom_2021', 'top_industry_mom_2022'
]

tables = get_tables(conn, list_tables)

# Inject into global variables
globals().update(tables)

In [7]:
# Here we load the table names into dataframes
sale_distribution_naics = tables['sale_distribution_naics']
sales_above_10b = tables['sales_above_10b']
outliers = tables['outliers']
kob_industry_total_sales = tables['kob_industry_total_sales']
auto_sales_2022 = tables['auto_sales_2022']
seasonality_mom = tables['seasonality_mom']
top_industry_mom_2019 = tables['top_industry_mom_2019']
top_industry_mom_2020 = tables['top_industry_mom_2020']
top_industry_mom_2021 = tables['top_industry_mom_2021']
top_industry_mom_2022 = tables['top_industry_mom_2022']

In [8]:
top_industry_mom_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   year         12 non-null     int64
 1   month        12 non-null     int64
 2   industry     12 non-null     str  
 3   total_sales  12 non-null     int64
dtypes: int64(3), str(1)
memory usage: 516.0 bytes


In [9]:
top_industry_mom_2022.head()

,year,month,industry,total_sales
0,2022,1,Automotive,420473
1,2022,2,Automotive,431998
2,2022,3,Automotive,514582
3,2022,4,Automotive,504116
4,2022,5,Automotive,483482
